# New Responses Examples 🍿

Let's do a quick snack-sized walkthrough of chatsnack's Responses runtime.

This notebook is intentionally beginner-friendly and follows the flavor of the existing chatsnack notebooks.


## Setup

### Got snack?
Install chatsnack and make sure your OpenAI SDK supports the Responses API (`openai>=1.66.0`).


In [ ]:
!pip install -q chatsnack "openai>=1.66.0" python-dotenv


### Got keys?
Put your API key in a local `.env` file:

```
OPENAI_API_KEY=sk-...
```


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from chatsnack import Chat
from chatsnack.chat.mixin_params import ChatParams


## Classic runtime (chat completions)
By default, chatsnack uses the chat completions runtime.


In [ ]:
classic = Chat("You are a concise tutor.", model="gpt-4.1-mini")
print("Runtime adapter:", type(classic.runtime).__name__)
classic.ask("Explain prompt engineering in one sentence.")


## New runtime (responses)
Now switch to the new Responses runtime with one argument.


In [ ]:
fresh = Chat(
    "You are a concise tutor.",
    runtime_selector="responses",
    model="gpt-4.1-mini",
)
print("Runtime adapter:", type(fresh.runtime).__name__)
fresh.ask("Explain prompt engineering in one sentence.")


### Same runtime choice via `ChatParams`
Useful when you want runtime selection and model settings bundled together.


In [ ]:
params = ChatParams(runtime="responses", model="gpt-4.1-mini", temperature=0.2)
teacher = Chat("You teach with short examples.", params=params)
teacher.ask("Give me a three-step checklist for my first prompt.")


## Speed taste test: old vs new
This quick benchmark compares *rough end-to-end latency* between runtimes.

> Note: network and model load vary. Run a few times and compare medians, not one-off results.


In [ ]:
from time import perf_counter
from statistics import median

def sample_latency(runtime_selector=None, n=3):
    timings = []
    for _ in range(n):
        chat = Chat(
            "Reply in exactly seven words.",
            runtime_selector=runtime_selector,
            model="gpt-4.1-mini",
            temperature=0,
        )
        t0 = perf_counter()
        chat.ask("Say hi to a new chatsnack user.")
        timings.append(perf_counter() - t0)
    return timings

classic_times = sample_latency(runtime_selector=None, n=3)
responses_times = sample_latency(runtime_selector="responses", n=3)

classic_med = median(classic_times)
responses_med = median(responses_times)
speedup = classic_med / responses_med if responses_med else float('inf')

print(f"classic median:   {classic_med:.3f}s -> {classic_times}")
print(f"responses median: {responses_med:.3f}s -> {responses_times}")
print(f"speedup (classic / responses): {speedup:.2f}x")


## Continue the conversation
`ask()` is a quick one-shot. `chat()` gives you a new chat with history carried forward.


In [ ]:
follow_up = teacher.chat("Now rewrite that checklist for a travel planner assistant.")
follow_up.last


## Optional: stream output
If you want token-style output, iterate `listen(...)`.


In [ ]:
for chunk in teacher.listen("Give me five tiny prompt-writing tips."):
    print(chunk, end="")
print()


---
You're ready to snack with the new Responses runtime.
